This is my locally adapted runtime because of library conflicts with my 5060 gpu

```bash
# Creating a custom environment
uv venv .venv-gpu # with python 3.12
source .venv-gpu/bin/activate # activate the environment
uv pip install tf-nightly[and-cuda] pip ipykernel pandas keras # install the libraries 
```
update
this tensorflow version works: tf-nightly[and-cuda]==2.21.0.dev20251017


In [1]:
import os
import re

import keras # Used for defining an training the model.
import pandas as pd # Used for loading the dataset.
import tensorflow as tf # Used for shuffling the dataset.
from IPython.display import display, HTML


# The following line provides configuration for Keras.
keras.utils.set_random_seed(128)  # For Keras layers.

I0000 00:00:1781168678.476577  320559 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781168678.503749  320559 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781168679.249846  320559 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# loading dataset

africa_galore = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore.json"
)
dataset = africa_galore["description"].values
print("Loaded dataset with", dataset.shape[0], "paragraphs.")

Loaded dataset with 232 paragraphs.


In [3]:
#  tokenization

class SimpleWordTokenizer:
    """A simple word tokenizer.

    The tokenizer splits the text sequence based on whitespace, using the
    `encode` method to convert the text into a sequence of indices and the
    `decode` method to convert indices back into text.

    The simple word tokenizer that can be initialized with a corpus or using a
    provided vocabulary list

    Typical usage example:

        corpus = "Hello there!"
        tokenizer = SimpleWordTokenizer(text)
        print(tokenizer.encode('Hello'))

    """

    # Define constants.
    UNKNOWN_TOKEN = "<UNK>"
    PAD_TOKEN = "<PAD>"

    def __init__(self, corpus: list[str], vocabulary: list[str] | None = None):
        """Initializes the tokenizer with texts in corpus or with a vocabulary.

        Args:
          corpus: Input text dataset.
          vocabulary: A pre-defined vocabulary. If None,
              the vocabulary is automatically inferred from the texts.
        """

        if vocabulary is None:
            # Build the vocabulary from scratch.
            if isinstance(corpus, str):
                corpus = [corpus]

            # Convert text sequence to tokens.
            tokens = []
            for text in corpus:
                for token in self.space_tokenize(text):
                    tokens.append(token)

            # Create a vocabulary comprising of unique tokens.
            vocabulary = self.build_vocabulary(tokens)

            # Add special unknown and pad tokens to the vocabulary list.
            self.vocabulary = (
                [self.PAD_TOKEN] + vocabulary + [self.UNKNOWN_TOKEN]
            )

        else:
            self.vocabulary = vocabulary

        # Size of vocabulary.
        self.vocabulary_size = len(self.vocabulary)

        # Create token-to-index and index-to-token mappings.
        self.token_to_index = {}
        self.index_to_token = {}
        # Loop through all tokens in the vocabulary. enumerate automatically
        # assigns a unique index to each token.
        for index, token in enumerate(self.vocabulary):
            self.token_to_index[token] = index
            self.index_to_token[index] = token

        # Map the special tokens to their IDs.
        self.pad_token_id = self.token_to_index[self.PAD_TOKEN]
        self.unknown_token_id = self.token_to_index[self.UNKNOWN_TOKEN]

    def space_tokenize(self, text: str) -> list[str]:
        """Splits a given text on whitespace into tokens.

        Args:
            text: Text to split on whitespace.

        Returns:
            List of tokens after splitting `text`.
        """

        # Use re.split such that multiple spaces are treated as a single
        # separator.
        return re.split(" +", text)

    def join_text(self, text_list: list[str]) -> str:
        """Combines a list of tokens into a single string.

        The combined tokens, as a single string, are separated by spaces in the
        string.

        Args:
            text_list: List of tokens to be joined.

        Returns:
            String with all tokens joined with a whitespace.

        """
        return " ".join(text_list)

    def build_vocabulary(self, tokens: list[str]) -> list[str]:
        """Create a vocabulary list from the list of tokens.

        Args:
            tokens: The list of tokens in the dataset.

        Returns:
            List of unique tokens (vocabulary) in the dataset.
        """
        return sorted(list(set(tokens)))

    def encode(self, text: str) -> list[int]:
        """Encodes a text sequence into a list of indices.

        Args:
            text: The input text to be encoded.

        Returns:
            A list of indices corresponding to the tokens in the input text.
        """

        # Convert tokens into indices.
        indices = []
        unk_index = self.token_to_index[self.UNKNOWN_TOKEN]
        for token in self.space_tokenize(text):
            token_index = self.token_to_index.get(token, unk_index)
            indices.append(token_index)

        return indices

    def decode(self, indices: int | list[int]) -> str:
        """Decodes a list (or single index) of integers back into tokens.

        Args:
            indices: A single index or a list of indices to be
                decoded into tokens.

        Returns:
            A string of decoded tokens corresponding to the input indices.
        """

        # If a single integer is passed, convert it into a list.
        if isinstance(indices, int):
            indices = [indices]

        # Map indices to tokens.
        tokens = []
        for index in indices:
            token = self.index_to_token.get(index, self.unknown_token_id)
            tokens.append(token)

        # Join the decoded tokens into a single string.
        return self.join_text(tokens)


# Initialize the tokenizer. This will build the tokenizer's vocabulary with
# all the tokens that appear in the dataset.
tokenizer = SimpleWordTokenizer(dataset)

# Translate all tokens to their corresponding IDs.
encoded_tokens = []
for text in dataset:
    # Split text into tokens and translate the tokens to token IDs.
    token_ids = tokenizer.encode(text)
    encoded_tokens.append(token_ids)

In [4]:
#  computing length of shortest and longest sentence
shortest_paragraph_length =len(min([encoded_token for encoded_token in encoded_tokens], key=len))

longest_paragraph_length = len(max([encoded_token for encoded_token in encoded_tokens], key=len))

print(f"Length of the shortest paragraph is:", shortest_paragraph_length)
print(f"Length of the longest paragraph is:", longest_paragraph_length)


Length of the shortest paragraph is: 26
Length of the longest paragraph is: 318


In [5]:
# padding the dataset
# set `max_length` for padding and truncating data.

max_length = 301 

if max_length <= 0:
    display(
        HTML(
            f"<h3>Error:</h3><p>Max length must be greater than 0. Please"
            f" increase <code>max_length</code>.</p><p></p>"
        )
    )

elif max_length > longest_paragraph_length:
    display(
        HTML(
            f"<h3>Error:</h3><p>The padding token <code>"
            f" {tokenizer.pad_token_id}</code> will be added to all"
            f" sequences - you probably don't want that. Please reduce"
            f" <code>max_length</code>.</p><p></p>"
        )
    )

else:
    if max_length < longest_paragraph_length:
        display(
            HTML(
                f"<p><strong>Note:</strong> The longest paragraph has"
                f" {longest_paragraph_length} tokens,"
                f" but <code>max_length</code> is set to {max_length}."
                f" Paragraphs longer than <code>max_length</code> will be"
                " truncated.</p><p></p>"
            )
        )

    padded_sequences = keras.preprocessing.sequence.pad_sequences(
        encoded_tokens,
        maxlen=max_length,
        padding="post",
        truncating="post",
        value=tokenizer.pad_token_id,
    )

    print("New length of first paragraph:", len(padded_sequences[0]), "\n")

    print(
        "Padding makes the length of all sequences the same as the specified"
        " `max_length`."
    )

    print(
        "Notice the padded token IDs {tokenizer.pad_token_id} appearing at the"
        f" end of the sequence.\n"
    )
    print("Padded tokens of first paragraph:\n", padded_sequences[0])

New length of first paragraph: 301 

Padding makes the length of all sequences the same as the specified `max_length`.
Notice the padded token IDs {tokenizer.pad_token_id} appearing at the end of the sequence.

Padded tokens of first paragraph:
 [ 814  511  985 5092 4802 5183 2800 1363 4792 2134 2856 4792 1584 5092
 2088  814 1134 3043 2922  912 2821  170 2623 4792 2023 3807 3576  912
 1653 3772 4792 2775 1244  912 4409 3280 1030 4792 1158 3049 1992  912
 1868 2486 2437  135 5189 3422  445 3388 2078 4849 4792 3407 2706 1259
 4692 2856 4839 5183 4792 4078  814 3406 4259 4849 2389 4831 2707  912
 3821 1829 3522 2134 1030 2955  185 1076 2707 3683 5143 1849 4343 1030
 1546 1446 4983 2856 4792 2876 4078  814 3406 5092 3366 4788 2968 2151
 2938 5092  912 1450 3522 3101  912 1672 4849 4793 4295 2721  912 5036
 2224 3522 4792 4437 3522  513    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    

In [6]:
# preparing input and target

# For each example, extract all tokens except the last one.
input_sequences = padded_sequences[:, :-1]
# For each example, extract all tokens except the first one.
target_sequences = padded_sequences[:, 1:]

print("First 10 token IDs in first input sequence:", input_sequences[0, :10])
print(
    "First 10 tokens in first input sequence:",
    tokenizer.decode(input_sequences[0, :10]),
)

print("\n")

print("First 10 token IDs in first target sequence:", target_sequences[0, :10])
print(
    "First 10 tokens in target sequence:",
    tokenizer.decode(target_sequences[0, :10])
)

First 10 token IDs in first input sequence: [ 814  511  985 5092 4802 5183 2800 1363 4792 2134]
First 10 tokens in first input sequence: The Lagos air was thick with humidity, but the energy


First 10 token IDs in first target sequence: [ 511  985 5092 4802 5183 2800 1363 4792 2134 2856]
First 10 tokens in target sequence: Lagos air was thick with humidity, but the energy in


In [7]:
# shuffling the dataset and specifying the batch size

# Create TensorFlow dataset to prepare sequences.
tf_dataset = tf.data.Dataset.from_tensor_slices((input_sequences, target_sequences))

# Randomly shuffle the dataset.
# The buffer_size determines how many examples from the dataset
# are held in memory before shuffling.
# If you are working with a very large dataset,
# reduce the buffer_size as needed.
tf_dataset = tf_dataset.shuffle(buffer_size=len(input_sequences))

# Specify batch size.
batch_size = 32  # @param {type: "number"}

# Create batches.
batches = tf_dataset.batch(batch_size)

for batch in batches.take(1):
    print(batch)

W0000 00:00:1781168680.537699  320559 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1781168680.544438  320559 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1781168680.625269  320559 gpu_device.cc:2040] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5796 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a


(<tf.Tensor: shape=(32, 300), dtype=int32, numpy=
array([[ 175, 1063,  876, ...,    0,    0,    0],
       [ 814,  358,  882, ...,    0,    0,    0],
       [ 477, 2932,  912, ...,    0,    0,    0],
       ...,
       [ 448, 4792, 1361, ...,    0,    0,    0],
       [ 323,  872, 2932, ...,    0,    0,    0],
       [ 607, 1023,  963, ...,    0,    0,    0]],
      shape=(32, 300), dtype=int32)>, <tf.Tensor: shape=(32, 300), dtype=int32, numpy=
array([[1063,  876,  163, ...,    0,    0,    0],
       [ 358,  882, 2932, ...,    0,    0,    0],
       [2932,  912, 3775, ...,    0,    0,    0],
       ...,
       [4792, 1361, 1540, ...,    0,    0,    0],
       [ 872, 2932,  912, ...,    0,    0,    0],
       [1023,  963, 5183, ...,    0,    0,    0]],
      shape=(32, 300), dtype=int32)>)


I0000 00:00:1781168681.195187  320559 local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [8]:
# get total number of batches

total_batches = 0
for batch in batches:
    total_batches += 1
print("Total number of batches is:", total_batches)

Total number of batches is: 8


I0000 00:00:1781168681.205604  320559 local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
# initializing the model
# uv pip install -U "jax[cuda13]"

from custom_ai_fndtns import training

model = training.create_model(
    max_length=max_length -1,
    vocabulary_size=tokenizer.vocabulary_size,
    learning_rate=1e-4
)

E0611 12:04:41.351377  320559 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 5.64GiB (6061572096 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0611 12:04:41.351421  320559 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 5.08GiB (5455414784 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0611 12:04:41.351456  320559 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 4.57GiB (4909873152 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0611 12:04:41.351491  320559 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 4.12GiB (4418885632 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0611 12:04:41.351527  320559 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 3.70GiB (3976996864 bytes) of de

In [10]:
prompt = "Abeni,"
prompt_ids = tokenizer.encode(prompt)
text_gen_callback = training.TextGenerator(
    max_tokens=10, start_tokens=prompt_ids, tokenizer=tokenizer, print_every=10
)



In [11]:
num_epochs = 200  # @param {type: "number"}
# verbose=2: Instructs the model.fit method to print one line per
# epoch so you see how the loss is decreasing and generated texts improving.
history = model.fit(
    x=batches, verbose=2, epochs=num_epochs, callbacks=[text_gen_callback]
)

Epoch 1/200


I0000 00:00:1781168683.279113  320731 service.cc:158] XLA service 0x7fcac003ab30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781168683.279127  320731 service.cc:166]   StreamExecutor device (0): NVIDIA GeForce RTX 5060 Laptop GPU, Compute Capability 12.0a
I0000 00:00:1781168683.310260  320731 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1781168683.384547  320731 assert_op.cc:39] Ignoring Assert operator compile_loss/custom_mask_pad_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1781168683.480524  320731 cuda_dnn.cc:463] Loaded cuDNN version 92301
I0000 00:00:1781168684.143537  321130 subprocess_compilation.cc:347] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 44 bytes spill stores, 28 bytes spill loads

I0000 00:00:1781168684.311676  321140 sub

8/8 - 20s - 2s/step - loss: 8.5758
Epoch 2/200
8/8 - 0s - 17ms/step - loss: 8.3571
Epoch 3/200
8/8 - 0s - 16ms/step - loss: 8.1543
Epoch 4/200
8/8 - 0s - 16ms/step - loss: 7.9710
Epoch 5/200
8/8 - 0s - 16ms/step - loss: 7.8037
Epoch 6/200
8/8 - 0s - 16ms/step - loss: 7.6529
Epoch 7/200
8/8 - 0s - 19ms/step - loss: 7.5147
Epoch 8/200
8/8 - 0s - 17ms/step - loss: 7.3912
Epoch 9/200
8/8 - 0s - 16ms/step - loss: 7.2758
Epoch 10/200


I0000 00:00:1781168704.006035  322788 subprocess_compilation.cc:347] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 64 bytes spill stores, 40 bytes spill loads

I0000 00:00:1781168704.027816  322792 subprocess_compilation.cc:347] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_9', 72 bytes spill stores, 76 bytes spill loads

I0000 00:00:1781168704.042237  322801 subprocess_compilation.cc:347] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5', 300 bytes spill stores, 304 bytes spill loads

I0000 00:00:1781168704.066784  322809 subprocess_compilation.cc:347] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1781168704.259469  322796 subprocess_compilation.cc:347] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_8', 8 bytes spill stores, 8 bytes spill lo

Generated text:
 Abeni, closer. matoke understanding portable absolute lowlands, thick northeastern influences nowhere 

8/8 - 3s - 396ms/step - loss: 7.1758
Epoch 11/200
8/8 - 0s - 16ms/step - loss: 7.0861
Epoch 12/200
8/8 - 0s - 16ms/step - loss: 7.0078
Epoch 13/200
8/8 - 0s - 16ms/step - loss: 6.9371
Epoch 14/200
8/8 - 0s - 16ms/step - loss: 6.8718
Epoch 15/200
8/8 - 0s - 16ms/step - loss: 6.8150
Epoch 16/200
8/8 - 0s - 16ms/step - loss: 6.7568
Epoch 17/200
8/8 - 0s - 17ms/step - loss: 6.6952
Epoch 18/200
8/8 - 0s - 17ms/step - loss: 6.6439
Epoch 19/200
8/8 - 0s - 16ms/step - loss: 6.5842
Epoch 20/200
Generated text:
 Abeni, guided Their nutrients a tuna. face like change powder. evening 

8/8 - 1s - 82ms/step - loss: 6.5222
Epoch 21/200
8/8 - 0s - 16ms/step - loss: 6.4611
Epoch 22/200
8/8 - 0s - 17ms/step - loss: 6.3900
Epoch 23/200
8/8 - 0s - 17ms/step - loss: 6.3272
Epoch 24/200
8/8 - 0s - 17ms/step - loss: 6.2562
Epoch 25/200
8/8 - 0s - 16ms/step - loss: 6.1863
Epoch 26/200
8/8 

In [13]:
# uv pip install matplotlib plotly nbformat
import tensorflow as tf
tf.experimental.numpy.experimental_enable_numpy_behavior()

from custom_ai_fndtns import visualizations

prompt = "Abeni, went looking for a" #@param {type: "string"}
num_tokens_to_generate = 10 #@param {type: "number"}
generated_text, probs = training.generate_text(
    prompt,
    num_tokens_to_generate,
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id,
    sampling_mode="greedy" # To generate the highest probability generation.
)

print("Generated text:", generated_text)
print("\n")

visualizations.plot_next_token(probs[0], prompt=prompt, tokenizer=tokenizer)

Generated text: Abeni, went looking for a small coastal fog, created using recipes passed down to the




In [15]:
prompt = "Jide was thirsty so she went looking for a" #@param {type: "string"}
num_tokens_to_generate = 10 #@param {type: "number"}
generated_text, probs = training.generate_text(
    prompt,
    num_tokens_to_generate,
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id,
    sampling_mode="random",
)

print("Generated text:", generated_text)
print("\n")

visualizations.plot_next_token(probs[0], prompt=prompt, tokenizer=tokenizer)

Generated text: Jide was thirsty so she went looking for a would dogs. Today, time singing, and a market bonnet peppers,


